In [0]:
"""Descarga la serie telemétrica de ayer para todas las estaciones del inventario local."""

from __future__ import annotations

import json
import os
import time
from datetime import date, timedelta
from pathlib import Path
from typing import Any, Iterable, Optional

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://www.ana.gov.br/hidrowebservice"
LOGIN_ENDPOINT = "/EstacoesTelemetricas/OAUth/v1"
SERIE_TELEMETRICA_ENDPOINT = "/EstacoesTelemetricas/HidroinfoanaSerieTelemetricaAdotada/v2"
DEFAULT_TIMEOUT = 60

OUTPUT_DIR = Path("/Volumes/weather/raw/ana_volume/json/")
STATIONS_FILE = Path("/Volumes/weather/raw/ana_volume/estaciones_rio_uruguai_pluvio_fluvio.json")
INTERVALO_BUSCA = "DIAS_2"

USER_API_ANA = dbutils.widgets.get("USER_API_ANA")
PASS_API_ANA = dbutils.widgets.get("PASS_API_ANA")

SERIE_INTERVAL_CHOICES = ["DIAS_2"]


class AnaApiError(RuntimeError):
    """Errores específicos del webservice de la ANA."""


def create_session() -> requests.Session:
    retry = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[500, 502, 503, 504],
        allowed_methods=["GET"],
    )

    adapter = HTTPAdapter(max_retries=retry)

    session = requests.Session()
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    return session


def _obtener_credenciales() -> tuple[str, str]:
    usuario = USER_API_ANA
    password = PASS_API_ANA
    if not usuario or not password:
        raise AnaApiError(
            "Configura las variables USER_API_ANA y PASS_API_ANA en el entorno o en un archivo .env"
        )
    return usuario, password


def log_api_ana(session: Optional[requests.Session] = None) -> str:
    usuario, password = _obtener_credenciales()
    sess = session or create_session()
    url = f"{BASE_URL}{LOGIN_ENDPOINT}"
    headers = {"Identificador": usuario, "Senha": password}
    response = sess.get(url, headers=headers, timeout=DEFAULT_TIMEOUT)
    try:
        response.raise_for_status()
        payload: dict[str, Any] = response.json()
    except (ValueError, requests.RequestException) as exc:
        raise AnaApiError(f"Error autenticando con ANA: {exc}") from exc

    try:
        token = payload["items"]["tokenautenticacao"]
    except (KeyError, TypeError) as exc:
        raise AnaApiError("La respuesta de login no contiene el token esperado") from exc

    if not isinstance(token, str) or not token:
        raise AnaApiError("Token de autenticación vacío o inválido")
    return token


def consultar_hidroinfoana_serie_telem_adotada(
    token: str,
    *,
    session: Optional[requests.Session] = None,
    codigos_estacoes: list[int],
    tipo_filtro_data: str = "DATA_LEITURA",
    data_busca: Optional[str] = None,
    intervalo: str = "DIAS_30",
) -> list[dict[str, Any]]:
    if not codigos_estacoes:
        raise AnaApiError("Debe informar al menos un código de estación")
    if len(codigos_estacoes) > 10:
        raise AnaApiError("El endpoint admite hasta 10 códigos de estación")
    if intervalo not in SERIE_INTERVAL_CHOICES:
        raise AnaApiError(f"Intervalo inválido. Use uno de: {', '.join(SERIE_INTERVAL_CHOICES)}")

    sess = session or create_session()
    url = f"{BASE_URL}{SERIE_TELEMETRICA_ENDPOINT}"
    params: dict[str, Any] = {
        "Codigos_Estacoes": ",".join(str(codigo) for codigo in codigos_estacoes),
        "Tipo Filtro Data": tipo_filtro_data,
        "Range Intervalo de busca": intervalo,
    }
    if data_busca:
        params["Data de Busca (yyyy-MM-dd)"] = data_busca

    headers = {"Authorization": f"Bearer {token}"}
    response = sess.get(url, headers=headers, params=params, timeout=DEFAULT_TIMEOUT)
    try:
        response.raise_for_status()
        payload: dict[str, Any] = response.json()
    except (ValueError, requests.RequestException) as exc:
        raise AnaApiError(
            f"Error consultando HidroinfoanaSerieTelemetricaAdotada: {exc}"
        ) from exc

    items = payload.get("items")
    if not isinstance(items, list):
        raise AnaApiError("La respuesta no contiene una lista de registros")
    return items


def _chunked(seq: Iterable[int], size: int) -> Iterable[list[int]]:
    chunk: list[int] = []
    for item in seq:
        chunk.append(item)
        if len(chunk) == size:
            yield chunk
            chunk = []
    if chunk:
        yield chunk


def _load_station_codes(path: Path = STATIONS_FILE) -> list[int]:
    try:
        estaciones = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise AnaApiError(f"No se pudo parsear el JSON de estaciones ({path}): {exc}") from exc
    
    if not isinstance(estaciones, list):
        raise AnaApiError("El archivo de estaciones debe contener una lista de objetos")

    codigos: list[int] = []
    presentes: set[int] = set()
    for estacion in estaciones:
        if not isinstance(estacion, dict):
            continue
        codigo_raw = estacion.get("codigoestacao")
        try:
            codigo = int(codigo_raw)
        except (TypeError, ValueError):
            continue
        if codigo not in presentes:
            presentes.add(codigo)
            codigos.append(codigo)

    if not codigos:
        raise AnaApiError("No se pudieron extraer códigos de estación del archivo proporcionado")
    return codigos


def _filter_by_date(registros: list[dict[str, Any]], target_date: date) -> list[dict[str, Any]]:
    objetivo = target_date.isoformat()
    filtrados: list[dict[str, Any]] = []
    for registro in registros:
        timestamp = registro.get("Data_Hora_Medicao", "")
        fecha = timestamp.split(" ")[0] if timestamp else ""
        if fecha == objetivo:
            filtrados.append(registro)
    return filtrados


def _dedupe_por_estacao_hora(registros: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Elimina duplicados conservando la medición más reciente por estación/hora."""

    combinados: dict[tuple[str, str], dict[str, Any]] = {}
    for registro in registros:
        codigo = registro.get("codigoestacao")
        timestamp = registro.get("Data_Hora_Medicao")
        if not codigo or not timestamp:
            continue
        clave = (str(codigo), str(timestamp))
        existente = combinados.get(clave)
        if not existente:
            combinados[clave] = registro
            continue
        if registro.get("Data_Atualizacao", "") >= existente.get("Data_Atualizacao", ""):
            combinados[clave] = registro
    return list(combinados.values())


def fetch_daily_level() -> None:
    codigos_estaciones = _load_station_codes()
    target_date = date.today() - timedelta(days=1)
    filename = f"ANA_{target_date.strftime('%Y_%m_%d')}.json"
    filepath = OUTPUT_DIR / filename
    if filepath.exists():
        print(f"Ya existe el archivo {filename}")
        return

    session = create_session()

    token = log_api_ana(session=session)
    registros: list[dict[str, Any]] = []

    for batch in _chunked(codigos_estaciones, 5):  # reducido de 10 a 5
        print(f"Consultando estaciones {batch[0]}-{batch[-1]} (total lote: {len(batch)})")

        time.sleep(1)  # pequeño delay para evitar saturar API

        lote = consultar_hidroinfoana_serie_telem_adotada(
            token,
            session=session,
            codigos_estacoes=batch,
            data_busca=target_date.isoformat(),
            intervalo=INTERVALO_BUSCA,
        )

        registros.extend(_filter_by_date(lote, target_date))

    datos_a_guardar = _dedupe_por_estacao_hora(registros)

    filepath.write_text(
        json.dumps(datos_a_guardar, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(
        f"OK {filename}",
        f"registros_guardados={len(datos_a_guardar)}",
    )


try:
    fetch_daily_level()
except AnaApiError as exc:
    raise SystemExit(f"✗ {exc}")